In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the eight harmonic amplitudes. The red bars are your recipe and the
gray marks are the clarinet's, read off the DFT above. On the left, the
red curve is two periods of your waveform over the gray clarinet shape.
The audio card plays a three second note from your recipe.

In [ ]:
# hide
# autorun
F0 = 300.0                          # the clarinet's fundamental
CLARINET = np.array([1.00, 0.05, 0.51, 0.08,     # read off the DFT above
                     0.12, 0.01, 0.03, 0.01])
K = np.arange(1, 9)
T_WAVE = np.linspace(0.0, 2 / F0, 800, endpoint=False)   # two periods on screen
BASIS = np.sin(2 * np.pi * F0 * K[:, None] * T_WAVE[None, :])
SR = 44100
T_NOTE = np.arange(int(3.0 * SR)) / SR
ENV = np.interp(T_NOTE, [0.0, 0.08, 2.7, 3.0], [0.0, 1.0, 1.0, 0.0])

def wave(amps, BASIS=BASIS):
    y = amps @ BASIS
    peak = np.abs(y).max()
    return y / peak if peak > 0 else y

def figure():
    fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38],
                        horizontal_spacing=0.12)
    fig.add_scatter(x=T_WAVE * 1000, y=wave(CLARINET), mode="lines",
                    line=dict(color=STEEL, width=2.2), row=1, col=1)
    fig.add_scatter(x=T_WAVE * 1000, y=wave(CLARINET), mode="lines",
                    line=dict(color=RED, width=2), row=1, col=1)
    fig.add_bar(x=K, y=CLARINET, marker_color=RED, row=1, col=2)
    fig.add_scatter(x=K, y=CLARINET, mode="markers",
                    marker=dict(color=IRON, size=20, symbol="line-ew",
                                line=dict(color=IRON, width=3)), row=1, col=2)
    fig.update_xaxes(title_text="Time (ms)", fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-1.15, 1.15], title_text="Amplitude (normalized)",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(title_text="Harmonic k", tickvals=list(K), fixedrange=True,
                     row=1, col=2)
    fig.update_yaxes(range=[0, 1.1], title_text="Amplitude", fixedrange=True,
                     row=1, col=2)
    return fig

def controls(fig):
    sliders = [widgets.FloatSlider(description=f"Harmonic {k} ({k * F0:.0f} Hz)",
                                   min=0, max=1, value=float(a), step=0.01)
               for k, a in zip(K, CLARINET)]
    readout = widgets.HTML()

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def update(wave=wave, readout=readout, **amps):
        a = np.array([amps[f"a{k}"] for k in range(1, 9)])
        with fig.batch_update():
            fig.data[1].y = wave(a)
            fig.data[2].y = a
        listed = ", ".join(f"{v:.2f}" for v in a)
        readout.value = (f"<span style='font-size:0.85em;font-family:monospace'>"
                         f"harmonic_amps = [{listed}]</span>")

    widgets.interactive_output(update, {f"a{k}": s for k, s in zip(K, sliders)})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(K=K, F0=F0, T_NOTE=T_NOTE, ENV=ENV, SR=SR):
        amps = np.array([s.value for s in sliders])
        x = sum(a * np.sin(2 * np.pi * k * F0 * T_NOTE) for a, k in zip(amps, K) if a > 0)
        x = ENV * x if np.ndim(x) else np.zeros_like(T_NOTE)
        peak = np.abs(x).max()
        if peak > 0:
            x *= 0.125 / peak                 # about -18 dBFS, a safe level
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)


    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (*sliders,):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([*sliders, readout, out, gate])

icm_plotly.show(figure, controls)